# 00 - Setup (run first, in every notebook)

**Team UL - DIRISA SDC Teams Qualification | The Democratic Funnel (LGE 2026)**

Works unchanged in **Google Colab** and **locally (PyCharm / VS Code / judges)**.

| What | Where it lives |
|---|---|
| Code (`src/`, `notebooks/`, `app/`) | GitHub repo |
| Data, models, figures | Shared Drive folder `DIRISA_SDC` (Colab) - or `DIRISA_ROOT` / the repo's `data/` locally |

This notebook:
1. Finds the repo (clones/pulls it in Colab) and mounts Drive
2. Creates the folder structure (safe to re-run)
3. Installs missing packages (Colab only)
4. Loads shared code, imports and settings
5. Logs package versions (reproducibility)
6. Builds the raw-data inventory and checks the sources log

**One-time per person (Colab):** in Drive, right-click `DIRISA_SDC` -> *Organise* -> *Add shortcut* -> *My Drive*.
If the repo is private, add a GitHub token in Colab -> 🔑 *Secrets* as `GITHUB_TOKEN`.

**Rule:** never save work only in `/content/` - it is wiped when the session ends.

## 1. Locate the repo and mount Drive

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"   # <- team repo URL
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass                                # public repo, or no token set
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    # local: walk up from the notebook until we find src/config.py
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

## 2. Extra packages (Colab only)
Colab already has pandas, numpy, scikit-learn, statsmodels, lightgbm, geopandas, matplotlib and seaborn. Locally, use `pip install -r requirements.txt` once instead.

In [ ]:
if IN_COLAB:
    %pip install -q shap pdfplumber

## 3. Shared code, folders and settings
`autoreload` means changes to `src/` are picked up without restarting the runtime.

In [ ]:
%load_ext autoreload
%autoreload 2

from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

if config.IN_COLAB and not config.DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        "DIRISA_SDC not found in My Drive. Right-click the shared folder -> "
        "Organise -> Add shortcut -> My Drive, then re-run.")

config.make_dirs()
config.set_seed()
print(config.describe())

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid")

## 4. Log the environment
Saved to `environment.json` in the work root so results can be reproduced with the same versions.

In [ ]:
import platform, datetime
import importlib.metadata as md

PKGS = ["pandas", "numpy", "scikit-learn", "statsmodels", "lightgbm", "shap",
        "geopandas", "matplotlib", "seaborn", "openpyxl", "streamlit"]

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return "not installed"

env = {"logged": datetime.datetime.now().isoformat(timespec="seconds"),
       "environment": "colab" if IN_COLAB else "local",
       "python": platform.python_version(),
       "packages": {p: _ver(p) for p in PKGS}}
(config.WORK_ROOT / "environment.json").write_text(json.dumps(env, indent=2))
env

## 5. Raw-data inventory
One row per CSV file / Excel sheet, with the detected header row (IEC reports often have title rows above the real header). Re-run whenever new files land in `data/raw`.

In [ ]:
inv = inventory()
inv.to_csv(config.WORK_ROOT / "data_inventory.csv", index=False)
print(f"{inv['file'].nunique() if len(inv) else 0} files, {len(inv)} sheets/tables")
inv

## 6. Sources log check
`docs/sources.csv` lives in the repo and ships with the submission. Every source needs a URL and access date; the registration dashboard also needs its **snapshot date**.

In [ ]:
sources = pd.read_csv(config.SOURCES_LOG)
todo = sources[sources["url"].isna() | sources["accessed_date"].isna()]
print(f"{len(todo)} of {len(sources)} sources still need a URL and/or access date")
todo[["source_id", "url", "accessed_date", "notes"]]

## Team conventions
- **One notebook per stage, one owner:** `01_results_lge`, `02_registration`, `03_byelections_npe`, `04_census_boundaries`, `05_master_table`, `06_eda_funnel`, `07_model`, `08_evaluation`.
- Every notebook starts by running cells 1-3 of this notebook (or `%run 00_setup.ipynb` locally).
- Reusable functions go in `src/`, not copied between notebooks.
- Read from `data/raw` or `data/interim`; write to `data/interim` or `data/processed`. **Never overwrite raw files.**
- Standardise columns with `clean_columns()` straight after reading.
- Join key is the municipality code (`muni_code`, e.g. `LIM354`, `TSH`). Exclude district municipalities as rows; keep `district_code` as a column.
- Log every download in `docs/sources.csv` (URL, access date, snapshot date).
- Colab is last-save-wins: don't edit the same notebook at the same time. Push notebooks to GitHub daily (*File -> Save a copy in GitHub*).
- **Candidate lists (personal data):** run `drop_personal()` immediately, keep counts only, never commit or share them.